In [ ]:
#Drive connect
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Verify data
import os

root = "/content/drive/MyDrive/Alvi/MLMI-2024"
for path, dirs, files in os.walk(root):
    print(path, len(files))
    break  # just show top level


/content/drive/MyDrive/Alvi/MLMI-2024 0


Dataset Preprocess

In [ ]:
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# ======================================================
# CORRECT ROOT PATH
# ======================================================
RAW_DIR = "/content/drive/MyDrive/Alvi/MLMI-2024"
OUT_DIR = "/content/drive/MyDrive/Alvi/processed"

SPECIES = {
    "Aedes aegypti": 0,
    "Culex quinquefasciatus": 1
}

VIEWS = ["head", "abdomen", "siphon", "full body"]

def prepare():
    all_larvae = []

    # --------------------------------------------------
    # STEP 1: COLLECT ALL IMAGE REFERENCES WITH UNIQUE IDs
    # --------------------------------------------------
    for species in SPECIES:
        species_dir = os.path.join(RAW_DIR, species)

        if not os.path.isdir(species_dir):
            raise FileNotFoundError(f"Missing species folder: {species_dir}")

        # Get all images from one view (to determine number of larvae)
        reference_view = VIEWS[0]
        view_dir = os.path.join(species_dir, reference_view)

        if not os.path.isdir(view_dir):
            raise FileNotFoundError(f"Missing view folder: {view_dir}")

        images = sorted(os.listdir(view_dir))
        num_larvae = len(images)

        print(f"✓ {species}: {num_larvae} larvae")

        # Create unique larva IDs using species prefix
        species_prefix = "aedes" if "aegypti" in species else "culex"

        for idx in range(num_larvae):
            larva_id = f"{species_prefix}_larva_{idx:03d}"

            # Store info for each larva
            larva_data = {
                'species': species,
                'larva_id': larva_id,
                'position': idx  # Position in sorted list
            }
            all_larvae.append(larva_data)

    print(f"\n📊 Total larvae collected: {len(all_larvae)}")



    # --------------------------------------------------
    # STEP 2: LARVA-LEVEL SPLIT (NO LEAKAGE)
    # --------------------------------------------------
    # Create labels for stratification
    labels = [SPECIES[larva['species']] for larva in all_larvae]

    # First split: 70% train, 30% temp
    train_larvae, temp_larvae = train_test_split(
        all_larvae,
        test_size=0.3,
        stratify=labels,
        random_state=42
    )

    # Second split: 15% val, 15% test
    temp_labels = [SPECIES[larva['species']] for larva in temp_larvae]
    val_larvae, test_larvae = train_test_split(
        temp_larvae,
        test_size=0.5,
        stratify=temp_labels,
        random_state=42
    )

    splits = {"train": train_larvae, "val": val_larvae, "test": test_larvae}

    print(f"\n📊 Split Summary:")
    print(f"   Train: {len(train_larvae)} larvae")
    print(f"   Val: {len(val_larvae)} larvae")
    print(f"   Test: {len(test_larvae)} larvae")

    # Verify no leakage
    train_ids = set(l['larva_id'] for l in train_larvae)
    val_ids = set(l['larva_id'] for l in val_larvae)
    test_ids = set(l['larva_id'] for l in test_larvae)

    assert len(train_ids & val_ids) == 0, "Train-Val leakage detected!"
    assert len(train_ids & test_ids) == 0, "Train-Test leakage detected!"
    assert len(val_ids & test_ids) == 0, "Val-Test leakage detected!"
    print("   ✅ No data leakage verified")

    # --------------------------------------------------
    # STEP 3: CREATE OUTPUT DIRECTORIES AND COPY FILES
    # --------------------------------------------------
    for split_name, larvae_list in splits.items():
        for larva in larvae_list:
            species = larva['species']
            larva_id = larva['larva_id']
            position = larva['position']

            # Create output directory
            out_dir = os.path.join(OUT_DIR, split_name, species, larva_id)
            os.makedirs(out_dir, exist_ok=True)

            # Copy all 4 views using position-based matching
            for view in VIEWS:
                view_dir = os.path.join(RAW_DIR, species, view)

                # Get the image at this position
                images = sorted(os.listdir(view_dir))
                src_image = images[position]

                src = os.path.join(view_dir, src_image)
                dst = os.path.join(out_dir, f"{view}.jpg")
                shutil.copy(src, dst)

    print("\n✅ Dataset prepared successfully")
    print(f"📁 Output directory: {OUT_DIR}")

    # Print sample of what was created
    print("\n📋 Sample larvae created:")
    for split_name in ["train", "val", "test"]:
        split_path = os.path.join(OUT_DIR, split_name)
        sample_count = 0
        for species in SPECIES:
            species_path = os.path.join(split_path, species)
            if os.path.exists(species_path):
                larvae = os.listdir(species_path)
                if larvae and sample_count == 0:
                    print(f"   {split_name}/{species}/{larvae[0]}/")
                    sample_count += 1

# Execute the function
prepare()

✓ Aedes aegypti: 100 larvae
✓ Culex quinquefasciatus: 100 larvae

📊 Total larvae collected: 200

📊 Split Summary:
   Train: 140 larvae
   Val: 30 larvae
   Test: 30 larvae
   ✅ No data leakage verified

✅ Dataset prepared successfully
📁 Output directory: /content/drive/MyDrive/Alvi/processed

📋 Sample larvae created:
   train/Aedes aegypti/aedes_larva_000/
   val/Aedes aegypti/aedes_larva_001/
   test/Aedes aegypti/aedes_larva_041/


In [ ]:
import os
from collections import defaultdict

VIEWS = ["head", "abdomen", "siphon", "full body"]
SPECIES = ["Aedes aegypti", "Culex quinquefasciatus"]

print("🔍 DIAGNOSING DATASET STRUCTURE\n")

for species in SPECIES:
    print(f"\n{'='*60}")
    print(f"SPECIES: {species}")
    print('='*60)

    species_dir = os.path.join(RAW_DIR, species)

    for view in VIEWS:
        view_dir = os.path.join(species_dir, view)

        if os.path.isdir(view_dir):
            images = sorted(os.listdir(view_dir))
            print(f"\n{view.upper()}:")
            print(f"  Total images: {len(images)}")
            print(f"  First 5 files:")
            for img in images[:5]:
                print(f"    - {img}")
        else:
            print(f"\n{view.upper()}:  FOLDER NOT FOUND")

    # Check for timestamp overlap
    print(f"\n{'─'*60}")
    print("Checking timestamp overlap across views...")

    timestamps_by_view = {}
    for view in VIEWS:
        view_dir = os.path.join(species_dir, view)
        if os.path.isdir(view_dir):
            images = os.listdir(view_dir)
            timestamps = set([img.replace('.jpg', '').replace('.JPG', '').replace('.png', '').replace('.PNG', '')
                             for img in images])
            timestamps_by_view[view] = timestamps

    if len(timestamps_by_view) == 4:
        # Find intersection (timestamps that appear in all 4 views)
        common_timestamps = set.intersection(*timestamps_by_view.values())
        print(f"  ✓ Timestamps appearing in ALL 4 views: {len(common_timestamps)}")

        if len(common_timestamps) == 0:
            print("\n   PROBLEM: No matching timestamps across all views!")
            print("  Checking each pair of views:")
            for i, view1 in enumerate(VIEWS):
                for view2 in VIEWS[i+1:]:
                    if view1 in timestamps_by_view and view2 in timestamps_by_view:
                        overlap = timestamps_by_view[view1] & timestamps_by_view[view2]
                        print(f"    {view1} ∩ {view2}: {len(overlap)} matches")

🔍 DIAGNOSING DATASET STRUCTURE


SPECIES: Aedes aegypti

HEAD:
  Total images: 100
  First 5 files:
    - 20231107_112550.jpg
    - 20231107_113540.jpg
    - 20231107_113902.jpg
    - 20231107_114014.jpg
    - 20231107_114958.jpg

ABDOMEN:
  Total images: 100
  First 5 files:
    - 20231211_131111.jpg
    - 20231211_131132.jpg
    - 20231211_131207.jpg
    - 20231211_131248.jpg
    - 20231211_131334.jpg

SIPHON:
  Total images: 100
  First 5 files:
    - 20231107_113223.jpg
    - 20231107_114314.jpg
    - 20231107_121955.jpg
    - 20231107_122403.jpg
    - 20231107_122943.jpg

FULL BODY:
  Total images: 100
  First 5 files:
    - 20231108_113905.jpg
    - 20231108_113934.jpg
    - 20231108_114028.jpg
    - 20231108_114117.jpg
    - 20231108_114212.jpg

────────────────────────────────────────────────────────────
Checking timestamp overlap across views...
  ✓ Timestamps appearing in ALL 4 views: 0

  ⚠️ PROBLEM: No matching timestamps across all views!
  Checking each pair of views:
   

In [ ]:
import os

VIEWS = ["head", "abdomen", "siphon", "full body"]
SPECIES = ["Aedes aegypti", "Culex quinquefasciatus"]

print("📊 IMAGE COUNT PER VIEW\n")

for species in SPECIES:
    print(f"{species}:")
    species_dir = os.path.join(RAW_DIR, species)

    counts = {}
    for view in VIEWS:
        view_dir = os.path.join(species_dir, view)
        if os.path.isdir(view_dir):
            count = len([f for f in os.listdir(view_dir) if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))])
            counts[view] = count
            print(f"  {view}: {count} images")

    # Check if all counts are equal
    if len(set(counts.values())) == 1:
        print(f"  ✅ All views have SAME count → Position-based matching likely valid\n")
    else:
        print(f"  ⚠️ Different counts → Cannot use position-based matching\n")


📊 IMAGE COUNT PER VIEW

Aedes aegypti:
  head: 100 images
  abdomen: 100 images
  siphon: 100 images
  full body: 100 images
  ✅ All views have SAME count → Position-based matching likely valid

Culex quinquefasciatus:
  head: 100 images
  abdomen: 100 images
  siphon: 100 images
  full body: 100 images
  ✅ All views have SAME count → Position-based matching likely valid



In [ ]:
import cv2
import numpy as np
import os
import random
import shutil
from albumentations import (
    HorizontalFlip, VerticalFlip, RandomRotate90, RandomBrightnessContrast,
    RandomGamma, RandomCrop, Resize, ElasticTransform, CLAHE, GaussNoise, Compose
)
from albumentations.pytorch import ToTensorV2

# Set the augmentation pipeline
def augmentation_pipeline():
    return [
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        RandomRotate90(p=0.5),
        RandomBrightnessContrast(p=0.5),
        RandomGamma(p=0.5),
        ElasticTransform(p=0.5),
        CLAHE(p=0.5),  # Contrast Limited Adaptive Histogram Equalization
        GaussNoise(p=0.2),
        RandomCrop(width=256, height=256, p=0.5),  # Random crop to a smaller region
        Resize(256, 256),  # Resize to target size
    ]

# Apply augmentation to an image
def augment_image(image_path, out_dir, larva_id, species, view):
    image = cv2.imread(image_path)  # Read image using OpenCV
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

    # Apply augmentations
    transform = Compose(augmentation_pipeline())
    augmented = transform(image=image)
    augmented_image = augmented['image']  # Augmented image

    # Convert back to BGR and save
    augmented_image = cv2.cvtColor(augmented_image, cv2.COLOR_RGB2BGR)
    aug_image_path = os.path.join(out_dir, f"{larva_id}_{view}_augmented.jpg")
    cv2.imwrite(aug_image_path, augmented_image)

# Loop through the splits and apply augmentations
def apply_augmentation():
    for split_name in ["train", "val", "test"]:
        split_path = os.path.join(OUT_DIR, split_name)

        for species in SPECIES:
            species_path = os.path.join(split_path, species)
            for larva_id in os.listdir(species_path):
                larva_path = os.path.join(species_path, larva_id)
                if os.path.isdir(larva_path):
                    for view in VIEWS:
                        view_path = os.path.join(larva_path, f"{view}.jpg")
                        if os.path.exists(view_path):
                            # Augment the image
                            out_dir = os.path.join(larva_path, 'augmented')
                            os.makedirs(out_dir, exist_ok=True)
                            augment_image(view_path, out_dir, larva_id, species, view)
    print("✅ Image augmentation complete!")

apply_augmentation()

✅ Image augmentation complete!


In [ ]:
from PIL import Image
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle

# ======================================================
# CONFIGURATION
# ======================================================
PROCESSED_DIR = "/content/drive/MyDrive/Alvi/processed"
MODEL_DIR = "/content/drive/MyDrive/Alvi/models"
os.makedirs(MODEL_DIR, exist_ok=True)

SPECIES = {
    "Aedes aegypti": 0,
    "Culex quinquefasciatus": 1
}

VIEWS = ["head", "abdomen", "siphon", "full body"]
IMG_SIZE = (224, 224)  # Resize images to this size

# ======================================================
# FEATURE EXTRACTION
# ======================================================
def extract_color_histogram(img, bins=32):
    """Extract color histogram features from RGB channels"""
    features = []
    for i in range(3):  # RGB channels
        hist = cv2.calcHist([img], [i], None, [bins], [0, 256])
        hist = hist.flatten() / hist.sum()  # Normalize
        features.extend(hist)
    return np.array(features)

def extract_texture_features(img_gray):
    """Extract texture features using GLCM-inspired statistics"""
    # Calculate basic texture statistics
    mean = np.mean(img_gray)
    std = np.std(img_gray)

    # Calculate gradients
    sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)

    gradient_magnitude = np.sqrt(sobelx**2 + sobely**2)
    edge_mean = np.mean(gradient_magnitude)
    edge_std = np.std(gradient_magnitude)

    return np.array([mean, std, edge_mean, edge_std])

def extract_shape_features(img_gray):
    """Extract shape-based features using contours"""
    # Threshold the image
    _, binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) > 0:
        # Get the largest contour
        largest_contour = max(contours, key=cv2.contourArea)

        area = cv2.contourArea(largest_contour)
        perimeter = cv2.arcLength(largest_contour, True)

        # Avoid division by zero
        if perimeter > 0:
            circularity = 4 * np.pi * area / (perimeter ** 2)
        else:
            circularity = 0

        # Hu Moments (shape descriptors)
        moments = cv2.moments(largest_contour)
        hu_moments = cv2.HuMoments(moments).flatten()

        return np.concatenate([[area, perimeter, circularity], hu_moments])
    else:
        return np.zeros(10)  # Return zeros if no contours found

def extract_features_from_image(img_path):
    """Extract all features from a single image"""
    # Read image
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Warning: Could not read image at {img_path}. Returning zero features.")
        return np.zeros(110) # Return a fixed-size zero array if image cannot be read

    img = cv2.resize(img, IMG_SIZE)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Extract different feature types
    color_features = extract_color_histogram(img, bins=32)
    texture_features = extract_texture_features(img_gray)
    shape_features = extract_shape_features(img_gray)

    # Concatenate all features
    all_features = np.concatenate([color_features, texture_features, shape_features])

    return all_features

def load_and_extract_features(data_dir, split_name):
    """Load all images from a split and extract features (including augmented images)"""
    features_list = []
    labels_list = []
    larva_ids = []

    split_path = os.path.join(data_dir, split_name)

    print(f"\n🔬 Extracting features from {split_name} set...")

    for species_name, label in SPECIES.items():
        species_path = os.path.join(split_path, species_name)

        if not os.path.exists(species_path):
            continue

        larvae = sorted(os.listdir(species_path))

        for larva_id in tqdm(larvae, desc=f"  {species_name}"):

            larva_path = os.path.join(species_path, larva_id)
            augmented_path = os.path.join(larva_path, 'augmented')  # Path to augmented images

            # Extract features from all views (original and augmented images)
            view_features = []
            for view in VIEWS:
                # Load original image
                img_path = os.path.join(larva_path, f"{view}.jpg")
                features_original = np.zeros(110) # Initialize with placeholder
                if os.path.exists(img_path):
                    features_original = extract_features_from_image(img_path)
                else:
                    print(f"⚠️ Missing original image: {img_path}")

                # Load augmented image
                augmented_img_path = os.path.join(augmented_path, f"{larva_id}_{view}_augmented.jpg")
                features_augmented = np.zeros(110) # Initialize with placeholder
                if os.path.exists(augmented_img_path):
                    features_augmented = extract_features_from_image(augmented_img_path)
                else:
                    print(f"⚠️ Missing augmented image: {augmented_img_path}")

                view_features.append(features_original)
                view_features.append(features_augmented)

            # Concatenate features from all views (both original and augmented)
            combined_features = np.concatenate(view_features)

            features_list.append(combined_features)
            labels_list.append(label)
            larva_ids.append(larva_id)

    return np.array(features_list), np.array(labels_list), larva_ids

In [ ]:
# ======================================================
# MODEL TRAINING
# ======================================================
def train_models(X_train, y_train, X_val, y_val):
    """Train multiple classifiers"""

    # Define models
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'SVM (Linear)': SVC(kernel='linear', random_state=42, probability=True),
        'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    }

    trained_models = {}
    results = []

    print("\n" + "="*60)
    print("TRAINING MODELS")
    print("="*60)

    for name, model in models.items():
        print(f"\n📊 Training {name}...")

        # Train
        model.fit(X_train, y_train)

        # Predict on train and validation
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        # Calculate metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        val_precision = precision_score(y_val, y_val_pred, average='weighted')
        val_recall = recall_score(y_val, y_val_pred, average='weighted')
        val_f1 = f1_score(y_val, y_val_pred, average='weighted')

        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Val Accuracy:   {val_acc:.4f}")
        print(f"  Val Precision:  {val_precision:.4f}")
        print(f"  Val Recall:     {val_recall:.4f}")
        print(f"  Val F1-Score:   {val_f1:.4f}")

        # Store model and results
        trained_models[name] = model
        results.append({
            'Model': name,
            'Train Accuracy': train_acc,
            'Val Accuracy': val_acc,
            'Val Precision': val_precision,
            'Val Recall': val_recall,
            'Val F1-Score': val_f1
        })

    # Create results dataframe
    results_df = pd.DataFrame(results)

    return trained_models, results_df

In [ ]:
# ======================================================
# EVALUATION
# ======================================================
def evaluate_on_test(models, X_test, y_test, scaler):
    """Evaluate all models on test set"""

    print("\n" + "="*60)
    print("TEST SET EVALUATION")
    print("="*60)

    test_results = []

    for name, model in models.items():
        print(f"\n📊 Testing {name}...")

        y_pred = model.predict(X_test)

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        print(f"  Accuracy:  {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1:.4f}")

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)

        test_results.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'Confusion Matrix': cm
        })

        # Plot confusion matrix
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Aedes', 'Culex'],
                   yticklabels=['Aedes', 'Culex'])
        plt.title(f'{name} - Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(os.path.join(MODEL_DIR, f'{name.replace(" ", "_")}_confusion_matrix.png'))
        plt.close()

        # Print classification report
        print(f"\n  Classification Report:")
        print(classification_report(y_test, y_pred,
                                   target_names=['Aedes aegypti', 'Culex quinquefasciatus']))

    return pd.DataFrame(test_results)

In [ ]:
# ======================================================
# SAVE MODELS
# ======================================================
def save_models(models, scaler):
    """Save trained models and scaler"""
    print("\n Saving models...")

    for name, model in models.items():
        filename = os.path.join(MODEL_DIR, f"{name.replace(' ', '_')}.pkl")
        with open(filename, 'wb') as f:
            pickle.dump(model, f)
        print(f"  ✅ Saved: {filename}")

    # Save scaler
    scaler_file = os.path.join(MODEL_DIR, "scaler.pkl")
    with open(scaler_file, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"  ✅ Saved: {scaler_file}")


In [ ]:
# ======================================================
# MAIN EXECUTION
# ======================================================
def main():
    print(" MOSQUITO LARVAE CLASSIFICATION")
    print("="*60)

    # Load and extract features
    X_train, y_train, train_ids = load_and_extract_features(PROCESSED_DIR, "train")
    X_val, y_val, val_ids = load_and_extract_features(PROCESSED_DIR, "val")
    X_test, y_test, test_ids = load_and_extract_features(PROCESSED_DIR, "test")

    print(f"\n📊 Dataset Summary:")
    print(f"  Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"  Val:   {X_val.shape[0]} samples")
    print(f"  Test:  {X_test.shape[0]} samples")

    # Feature scaling
    print(f"\n🔧 Applying feature scaling...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Train models
    trained_models, val_results = train_models(X_train_scaled, y_train,
                                               X_val_scaled, y_val)

    # Display validation results
    print("\n" + "="*60)
    print("VALIDATION RESULTS SUMMARY")
    print("="*60)
    print(val_results.to_string(index=False))

    # Evaluate on test set
    test_results = evaluate_on_test(trained_models, X_test_scaled, y_test, scaler)

    # Display test results
    print("\n" + "="*60)
    print("TEST RESULTS SUMMARY")
    print("="*60)
    print(test_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

    # Save models
    save_models(trained_models, scaler)

    # Save results to CSV
    val_results.to_csv(os.path.join(MODEL_DIR, 'validation_results.csv'), index=False)
    test_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_csv(
        os.path.join(MODEL_DIR, 'test_results.csv'), index=False)

    print(f"\n✅ All done! Results saved to {MODEL_DIR}")

    # Find best model
    best_model_name = test_results.loc[test_results['Accuracy'].idxmax(), 'Model']
    best_accuracy = test_results['Accuracy'].max()
    print(f"\n🏆 Best Model: {best_model_name} (Test Accuracy: {best_accuracy:.4f})")

# Run the pipeline
if __name__ == "__main__":
    main()

🦟 MOSQUITO LARVAE CLASSIFICATION

🔬 Extracting features from train set...


  Culex quinquefasciatus: 100%|██████████| 70/70 [00:38<00:00,  1.80it/s]



🔬 Extracting features from val set...


  Culex quinquefasciatus: 100%|██████████| 15/15 [00:08<00:00,  1.72it/s]



🔬 Extracting features from test set...


  Culex quinquefasciatus: 100%|██████████| 15/15 [00:08<00:00,  1.74it/s]



📊 Dataset Summary:
  Train: 140 samples, 880 features
  Val:   30 samples
  Test:  30 samples

🔧 Applying feature scaling...

TRAINING MODELS

📊 Training Logistic Regression...
  Train Accuracy: 1.0000
  Val Accuracy:   1.0000
  Val Precision:  1.0000
  Val Recall:     1.0000
  Val F1-Score:   1.0000

📊 Training SVM (Linear)...
  Train Accuracy: 1.0000
  Val Accuracy:   1.0000
  Val Precision:  1.0000
  Val Recall:     1.0000
  Val F1-Score:   1.0000

📊 Training SVM (RBF)...
  Train Accuracy: 1.0000
  Val Accuracy:   0.9667
  Val Precision:  0.9688
  Val Recall:     0.9667
  Val F1-Score:   0.9666

📊 Training Random Forest...
  Train Accuracy: 1.0000
  Val Accuracy:   1.0000
  Val Precision:  1.0000
  Val Recall:     1.0000
  Val F1-Score:   1.0000

📊 Training XGBoost...
  Train Accuracy: 1.0000
  Val Accuracy:   1.0000
  Val Precision:  1.0000
  Val Recall:     1.0000
  Val F1-Score:   1.0000

VALIDATION RESULTS SUMMARY
              Model  Train Accuracy  Val Accuracy  Val Precision